In [16]:
import pandas as pd
import scanpy as sc

In [2]:
split = 0
adata_train_path = f"/home/haicu/soeren.becker/repos/ot_pert_reproducibility/norman2019/norman_preprocessed_adata/adata_train_pca_50_split_{split}.h5ad"
adata = sc.read_h5ad(adata_train_path)

In [3]:
adata.uns.keys()

dict_keys(['esm2', 'non_dropout_gene_idx', 'non_zeros_gene_idx', 'pca', 'rank_genes_groups_cov_all', 'top_non_dropout_de_20', 'top_non_zero_de_20'])

In [4]:
gene_names = list(adata.uns["esm2"].keys())
conv_dict = {"C19orf26": "CBARP", "C3orf72": "FOXL2NB", "ELMSAN1": "MIDEAS"}
gene_names = [el for el in gene_names if el not in conv_dict.keys()] + list(conv_dict.values())

In [5]:
len(gene_names)

103

In [6]:
import mygene

mg = mygene.MyGeneInfo()
results = mg.querymany(gene_names, scopes='symbol', fields='ensembl.gene', species='human')


Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed


In [7]:
gene_to_ensembl = {}
for el in results:
    gene = el["query"]
    to_append = el["ensembl"]
    if len(to_append) == 1:
        gene_to_ensembl[gene] = to_append["gene"]
    else:
        gene_to_ensembl[gene] = to_append[0]["gene"]
    


In [8]:
ensembl_to_gene = {v:k for k,v in gene_to_ensembl.items()}

In [9]:
ensembl_ids = gene_to_ensembl.values()

In [10]:
df_omics_256 = pd.read_csv("/lustre/groups/ml01/workspace/ot_perturbation/data/embeddings/omics_d256.tsv", sep="\t")

In [11]:
df_omics_256

,gene_id,FACT_EMB_0,FACT_EMB_1,FACT_EMB_2,FACT_EMB_3,FACT_EMB_4,FACT_EMB_5,FACT_EMB_6,FACT_EMB_7,FACT_EMB_8,...,FACT_EMB_246,FACT_EMB_247,FACT_EMB_248,FACT_EMB_249,FACT_EMB_250,FACT_EMB_251,FACT_EMB_252,FACT_EMB_253,FACT_EMB_254,FACT_EMB_255
0,ENSG00000000003,-1.193790,-0.038485,0.477347,-1.355883,-1.283844,0.577406,-0.085741,-0.141601,0.929133,...,1.202038,-0.015473,0.732149,-0.470897,0.163932,0.514095,-1.121097,0.353111,-0.375309,0.271749
1,ENSG00000000005,-0.678243,-0.082044,0.628727,-0.036960,-0.327301,-0.132137,-0.658378,0.382881,0.321670,...,0.158936,1.025258,-0.944454,-0.110598,0.059320,-0.250677,-0.524779,-0.302290,0.627002,0.164144
2,ENSG00000000419,0.763473,0.029731,0.157848,-1.063333,-1.121445,0.855230,0.054342,0.782584,0.679390,...,0.215326,-0.662693,0.154820,0.359877,-0.675248,-0.293355,-0.355871,0.380746,-0.834603,-0.782095
3,ENSG00000000457,0.500445,0.107114,0.758035,-1.097068,0.355529,0.676653,0.562473,-0.585225,0.680695,...,-0.443879,0.226092,0.258948,0.716421,0.022804,-0.488464,-0.700852,-0.285765,0.137165,-0.220267
4,ENSG00000000460,0.085451,0.300662,-0.520867,-0.111104,0.116884,1.271965,0.750282,-0.769611,0.652003,...,-0.781987,-0.538280,-1.022490,-0.261985,-0.225361,0.223187,-0.840165,1.048801,-0.219930,-0.667091
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19380,ENSG00000280178,0.151255,0.066157,0.085184,-0.053143,-0.270811,0.211693,-0.401942,0.048608,0.396584,...,0.036106,0.029502,-0.247412,-0.039378,-0.355921,-0.143174,-0.001159,0.033741,-0.136875,-0.467921
19381,ENSG00000280253,0.139209,0.173939,-0.025301,0.197056,-0.086932,0.248732,-0.317894,0.000547,0.264560,...,0.139947,-0.031823,-0.109894,0.097934,-0.243768,-0.176109,-0.002984,-0.027844,0.069011,-0.645361
19382,ENSG00000280267,-0.799880,-0.290801,0.281187,-0.347249,-0.004650,0.019152,0.336727,0.289013,-0.390199,...,-0.213539,-0.293435,-0.636833,0.811885,-0.119030,0.375083,0.600217,-0.772333,0.670361,-0.510163
19383,ENSG00000280297,0.125944,0.020628,-0.069116,-0.076246,-0.322970,0.216778,-0.389412,0.208437,0.143095,...,-0.069501,0.155821,-0.123458,0.270595,-0.460678,0.027504,0.098230,-0.058452,0.088184,-0.454670


In [12]:
import mygene

mg = mygene.MyGeneInfo()
ensembl_ids = list(df_omics_256["gene_id"].values)
results = mg.querymany(ensembl_ids, scopes='ensembl.gene', fields='symbol', species='human')

ensembl_to_symbol = {r['query']: r.get('symbol', None) for r in results}



Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
1 input query terms found dup hits:	[('ENSG00000268674', 3)]
181 input query terms found no hit:	['ENSG00000069712', 'ENSG00000112096', 'ENSG00000116957', 'ENSG00000122718', 'ENSG00000130201', 'ENS


In [13]:
df_omics_256["gene_name"] = df_omics_256["gene_id"].map(ensembl_to_symbol)

In [14]:
set(ensembl_ids) - set(df_omics_256["gene_id"].values)

set()

In [15]:
df_omics_256.head()

,gene_id,FACT_EMB_0,FACT_EMB_1,FACT_EMB_2,FACT_EMB_3,FACT_EMB_4,FACT_EMB_5,FACT_EMB_6,FACT_EMB_7,FACT_EMB_8,...,FACT_EMB_247,FACT_EMB_248,FACT_EMB_249,FACT_EMB_250,FACT_EMB_251,FACT_EMB_252,FACT_EMB_253,FACT_EMB_254,FACT_EMB_255,gene_name
0,ENSG00000000003,-1.193790,-0.038485,0.477347,-1.355883,-1.283844,0.577406,-0.085741,-0.141601,0.929133,...,-0.015473,0.732149,-0.470897,0.163932,0.514095,-1.121097,0.353111,-0.375309,0.271749,TSPAN6
1,ENSG00000000005,-0.678243,-0.082044,0.628727,-0.036960,-0.327301,-0.132137,-0.658378,0.382881,0.321670,...,1.025258,-0.944454,-0.110598,0.059320,-0.250677,-0.524779,-0.302290,0.627002,0.164144,TNMD
2,ENSG00000000419,0.763473,0.029731,0.157848,-1.063333,-1.121445,0.855230,0.054342,0.782584,0.679390,...,-0.662693,0.154820,0.359877,-0.675248,-0.293355,-0.355871,0.380746,-0.834603,-0.782095,DPM1
3,ENSG00000000457,0.500445,0.107114,0.758035,-1.097068,0.355529,0.676653,0.562473,-0.585225,0.680695,...,0.226092,0.258948,0.716421,0.022804,-0.488464,-0.700852,-0.285765,0.137165,-0.220267,SCYL3
4,ENSG00000000460,0.085451,0.300662,-0.520867,-0.111104,0.116884,1.271965,0.750282,-0.769611,0.652003,...,-0.538280,-1.022490,-0.261985,-0.225361,0.223187,-0.840165,1.048801,-0.219930,-0.667091,FIRRM


In [451]:
gene_names = adata.uns["esm2"].keys()

In [452]:
len(gene_names)

103

In [453]:
set(gene_names) - set(df_omics_256["gene_name"])

{'C19orf26', 'C3orf72', 'ELMSAN1', 'ctrl'}

In [454]:
'C19orf26' in set(df_omics_256["gene_name"].values)

False

In [455]:
conv_dict

{'C19orf26': 'CBARP', 'C3orf72': 'FOXL2NB', 'ELMSAN1': 'MIDEAS'}

In [463]:
df_omics_256["gene_name_adapted"] = df_omics_256["gene_name"].replace({v:k for k,v in conv_dict.items()})

In [464]:
df_omics_256[df_omics_256["gene_name"]=='CBARP']

,gene_id,FACT_EMB_0,FACT_EMB_1,FACT_EMB_2,FACT_EMB_3,FACT_EMB_4,FACT_EMB_5,FACT_EMB_6,FACT_EMB_7,FACT_EMB_8,...,FACT_EMB_248,FACT_EMB_249,FACT_EMB_250,FACT_EMB_251,FACT_EMB_252,FACT_EMB_253,FACT_EMB_254,FACT_EMB_255,gene_name,gene_name_adapted
2089,ENSG00000099625,-0.338605,1.503707,-0.898282,0.953295,-0.1444,0.424698,0.67428,-0.02668,-0.14126,...,-0.285988,-0.766475,-1.277687,-0.118616,-0.23464,0.219932,0.513566,-1.003213,CBARP,C19orf26


In [466]:
df_omics_reduced = df_omics_256[df_omics_256["gene_name_adapted"].isin(gene_names)]

In [467]:
df_omics_reduced = df_omics_reduced.set_index("gene_name_adapted")

In [468]:
df_omics_reduced.shape

(102, 258)

In [469]:
df_omics_reduced = df_omics_reduced.drop(["gene_id", "gene_name"], axis=1)

In [470]:
df_omics_reduced.head()

,FACT_EMB_0,FACT_EMB_1,FACT_EMB_2,FACT_EMB_3,FACT_EMB_4,FACT_EMB_5,FACT_EMB_6,FACT_EMB_7,FACT_EMB_8,FACT_EMB_9,...,FACT_EMB_246,FACT_EMB_247,FACT_EMB_248,FACT_EMB_249,FACT_EMB_250,FACT_EMB_251,FACT_EMB_252,FACT_EMB_253,FACT_EMB_254,FACT_EMB_255
gene_name_adapted,,,,,,,,,,,,,,,,,,,,,
SLC4A1,0.391509,-0.274131,-0.337939,0.129666,0.376928,-0.459088,-0.653308,-0.565502,0.948452,1.335584,...,-0.832930,0.198551,-0.175683,0.301552,0.281877,-0.204890,0.677471,-0.148238,0.664731,-0.581532
MAP4K3,1.128384,-0.001505,0.598662,-0.207837,-0.593468,0.518699,-0.409682,-0.217646,0.405821,0.033550,...,0.538499,-0.504540,0.209822,0.488854,0.154927,-0.159430,0.299696,-0.368628,0.000459,0.318937
MAP4K5,0.188387,0.369866,0.290838,-0.278204,-0.309085,1.199782,-0.583357,-0.318135,-0.767403,-0.858054,...,-0.249090,-0.848268,1.278003,0.033087,-0.563374,-0.017930,-0.156311,-0.908655,-0.345090,-0.154722
BAK1,0.689212,-0.278187,-1.380527,0.532915,0.393399,0.330163,0.501224,-0.809612,0.760095,-0.209523,...,-0.167518,0.063678,0.349543,-0.272489,-1.293536,-0.602938,0.516270,0.150707,0.601678,-0.977561
MAP2K3,1.059429,-0.144727,-0.558349,1.742335,0.920236,-0.053834,-0.265533,-1.070187,0.783938,0.986686,...,0.234546,0.566083,-0.635409,-0.481470,-0.169609,0.312294,0.320896,-0.241767,1.587820,-0.492876


In [471]:
omics_rep = {}
for row, arr in df_omics_reduced.iterrows():
    omics_rep[row] = df_omics_reduced.loc[row].values

In [473]:
omics_rep["ctrl"] = np.zeros((256,))

In [474]:
with open('/lustre/groups/ml01/workspace/ot_perturbation/data/embeddings/gene_nargab.pkl', 'wb') as f:
    pickle.dump(omics_rep, f)

In [425]:
set(gs) - set(df_omics_256["gene_name"].values)

{'C19orf26', 'C3orf72', 'ELMSAN1', 'ctrl'}

In [ ]:
conv_dict = {"C19orf26": "CBARP", "C3orf72": "FOXL2NB", "ELMSAN1": "MIDEAS"}

In [426]:
set(conv_dict.values()) - set(df_omics_256["gene_name"].values)

set()

# PRESAGE

In [17]:
import pandas 
import pickle
import pandas as pd
import os
path = "/lustre/groups/ml01/workspace/ot_perturbation/data/embeddings/cache/pathway_embeddings"

In [18]:
gene_embs = ["c7.immunesigdb.v2023.2.Hs.symbols.pkl",
"c7.all.v2023.2.Hs.symbols.pkl",
"c3.all.v2023.2.Hs.symbols.pkl", # (this has all)
"c5.go.v2023.2.Hs.symbols.pkl", #(this has all)
"c1.all.v2023.2.Hs.symbols.pkl", # (this has all)
"c5.go.bp.v2023.2.Hs.symbols.pkl", 
"c3.tft.v2023.2.Hs.symbols.pkl", # (this has all)
"c3.tft.gtrd.v2023.2.Hs.symbols.pkl",
"c2.cgp.v2023.2.Hs.symbols.pkl",
"stringdb.human.medium.pkl"]

In [19]:
alias_dict = {"FOXL2NB": ["C3orf72", "C3ORF72", "FLJ43329", "FOXL2"],
             "CBARP": ["BARP", "DOS", "C19orf26", "C19ORF26"],
             "SAMD1": [],
             "CITED1": ["MSG1"],
             "CLDN6": ["Claudin 6", "Claudin-6"],
             "CEBPE": ["CRP1"],
             "MIDEAS": ["C14orf117", "C14orf43", "ELMSAN1", "LSR68"],
             }

In [20]:
for emb in gene_embs:
    df = pd.read_pickle(os.path.join(path, emb))
    diff = set(ensembl_to_gene.values()) - set(df.index)
    for el in diff:
        if el == "ctrl":
            continue
        aliases = alias_dict[el]
        print("aliases found: ", set(ensembl_to_gene.values()).intersection(set(aliases)))
    print(emb, diff-set(("ctrl", )))

aliases found:  {'FOXL2'}
c7.immunesigdb.v2023.2.Hs.symbols.pkl {'FOXL2NB'}
aliases found:  {'FOXL2'}
c7.all.v2023.2.Hs.symbols.pkl {'FOXL2NB'}
c3.all.v2023.2.Hs.symbols.pkl set()
c5.go.v2023.2.Hs.symbols.pkl set()
c1.all.v2023.2.Hs.symbols.pkl set()
aliases found:  {'FOXL2'}
c5.go.bp.v2023.2.Hs.symbols.pkl {'FOXL2NB'}
c3.tft.v2023.2.Hs.symbols.pkl set()
aliases found:  set()
c3.tft.gtrd.v2023.2.Hs.symbols.pkl {'CEBPE'}
aliases found:  {'FOXL2'}
c2.cgp.v2023.2.Hs.symbols.pkl {'FOXL2NB'}
aliases found:  set()
stringdb.human.medium.pkl {'MIDEAS'}


In [21]:
aliases

['C14orf117', 'C14orf43', 'ELMSAN1', 'LSR68']

In [22]:
dfs = {}
genes = list(set(ensembl_to_gene.values()) - set(("ctrl", )))
for emb in gene_embs:
    df = pd.read_pickle(os.path.join(path, emb))
    diff = set(genes) - set(df.index)
    df_red = df[df.index.isin(ensembl_to_gene.values())]
    dfs_to_append = []
    for el in diff:
        aliases = alias_dict[el]
        aliases = list(set(aliases).intersection(set(genes)))
        if len(aliases) > 1:
            print(el, aliases)
            raise ValueError
        if len(aliases) == 1:
            dff = df.loc[[aliases[0]]]
            dff.index = [el]
            dfs_to_append.append(dff)
        else:
            df_tmp = pd.DataFrame(df_red.mean(axis=0)).T
            df_tmp.index = [el]
            dfs_to_append.append(df_tmp)

    if len(dfs_to_append):
        df_all = pd.concat((df_red, pd.concat(dfs_to_append, axis=0)), axis=0)
        df_all2 = df_all.copy()
    else:
        df_all = df_red
    dfs[emb] = df_all.loc[genes]
        


In [23]:
for df in dfs.values():
    print(df.shape)

(102, 128)
(102, 128)
(102, 128)
(102, 128)
(102, 128)
(102, 128)
(102, 128)
(102, 128)
(102, 128)
(102, 128)


In [26]:
df_all = pd.concat(dfs,axis=1)

In [27]:
df_all.shape

(102, 1280)

In [31]:
set(adata.uns['esm2'].keys())- set(df_all.index)

{'C19orf26', 'C3orf72', 'ELMSAN1', 'ctrl'}

In [32]:
set(df_all.index) - set(adata.uns['esm2'].keys())

{'CBARP', 'FOXL2NB', 'MIDEAS'}

In [34]:
set(ensembl_to_gene.values()) - set(df_all.index) 

{'ctrl'}

In [41]:
alias_dict = {"C3orf72": ["FOXL2NB", "C3ORF72", "FLJ43329", "FOXL2"],
             "C19orf26": ["BARP", "DOS", "CBARP", "C19ORF26"],
             "SAMD1": [],
             "CITED1": ["MSG1"],
             "CLDN6": ["Claudin 6", "Claudin-6"],
             "CEBPE": ["CRP1"],
             "ELMSAN1": ["C14orf117", "C14orf43", "MIDEAS", "LSR68"],
             }

In [42]:
genes = list(set(adata.uns['esm2'].keys()) - set(("ctrl", )))
dfs = {}
for emb in gene_embs:
    df = pd.read_pickle(os.path.join(path, emb))
    diff = set(genes) - set(df.index)
    df_red = df[df.index.isin(genes)]
    dfs_to_append = []
    for el in diff:
        aliases = alias_dict[el]
        aliases = list(set(aliases).intersection(set(genes)))
        if len(aliases) > 1:
            print(el, aliases)
            raise ValueError
        if len(aliases) == 1:
            dff = df.loc[[aliases[0]]]
            dff.index = [el]
            dfs_to_append.append(dff)
        else:
            df_tmp = pd.DataFrame(df_red.mean(axis=0)).T
            df_tmp.index = [el]
            dfs_to_append.append(df_tmp)

    if len(dfs_to_append):
        df_all = pd.concat((df_red, pd.concat(dfs_to_append, axis=0)), axis=0)
        df_all2 = df_all.copy()
    else:
        df_all = df_red
    dfs[emb] = df_all.loc[genes]
        


In [43]:
df_all = pd.concat(dfs,axis=1)

In [45]:
df_dict = {idx: row.to_numpy() for idx, row in df_all.iterrows()}


In [49]:
import numpy as np
df_dict["ctrl"] = np.zeros_like(next(iter(df_dict.values())))

In [50]:
with open('/lustre/groups/ml01/workspace/ot_perturbation/data/embeddings/gene_mix.pkl', 'wb') as f:
    pickle.dump(df_dict, f)